# Block 2 (conditions) — stimulus-text checks

Verifies `pipeline/extract_conditions.py` / `pipeline/conditions.json`: the
intervention stimulus texts Block 4 will inject into each respondent's prompt.

Checks:
1. **All 16 interventions** extracted, labels matching the schema exactly.
2. **Control** carries no stimulus (it shows neutral off-topic filler).
3. **Every stimulus is non-trivial** (non-empty, sane length).
4. **No layout junk** left in any stimulus (`###`, page-break markers, dividers).
5. **Provenance** — each stimulus actually appears in `survey/questionnaire.txt`
   (guards against extraction bugs / fabrication).
6. **Cross-check vs R** — labels == `submission_spec.R` interventions (skips if no R).

In [ ]:
import json, re, shutil, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "codebook.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "codebook.csv").exists(), f"can't find repo root from cwd={Path.cwd()}"
PIPE = ROOT / "pipeline"
sys.path.insert(0, str(PIPE))
print("repo root:", ROOT)

failures = []
def check(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  -- {detail}" if (detail and not ok) else ""))
    if not ok:
        failures.append((name, detail))

## Step 0 — (Re)build and load conditions + schema

In [ ]:
import extract_conditions
extract_conditions.main()                                  # regenerate conditions.json
cond   = json.loads((PIPE / "conditions.json").read_text(encoding="utf-8"))
schema = json.loads((PIPE / "schema.json").read_text(encoding="utf-8"))
interventions = cond["interventions"]
print(f"{len(interventions)} interventions; control stimulus = {cond['control']['stimulus_text']!r}")

## Check 1 — All 16 interventions, labels match the schema

In [ ]:
schema_iv = set(schema["conditions"]["interventions"])
check("16 interventions extracted", len(interventions) == 16, f"got {len(interventions)}")
check("intervention labels == schema interventions", set(interventions) == schema_iv,
      f"only_in_conditions={set(interventions)-schema_iv} | only_in_schema={schema_iv-set(interventions)}")

## Check 2 — Control carries no stimulus

In [ ]:
check("control.is_control == True", cond["control"].get("is_control") is True)
check("control.stimulus_text is None", cond["control"]["stimulus_text"] is None)

## Check 3 — Every stimulus is non-trivial

In [ ]:
too_short = {k: v["char_len"] for k, v in interventions.items()
             if not v["stimulus_text"] or v["char_len"] < 150}
check("every intervention has a non-trivial stimulus (>=150 chars)", not too_short, str(too_short))

## Check 4 — No layout junk left in any stimulus

In [ ]:
junk = re.compile(r"###|page\s*break|^[=\-]{6,}$|TRANSITION", re.IGNORECASE | re.MULTILINE)
dirty = {k: junk.findall(v["stimulus_text"]) for k, v in interventions.items()
         if junk.search(v["stimulus_text"])}
check("no layout markers remain in stimulus texts", not dirty, str(dirty))

## Check 5 — Provenance: each stimulus appears in questionnaire.txt

In [ ]:
def norm(s): return re.sub(r"\s+", " ", s).strip()
qtxt = norm((ROOT / "survey" / "questionnaire.txt").read_text(encoding="utf-8"))
missing_prov = []
for k, v in interventions.items():
    snippet = norm(v["stimulus_text"])[:50]      # distinctive opening of the passage
    if snippet not in qtxt:
        missing_prov.append((k, snippet))
check("every stimulus traces back to questionnaire.txt", not missing_prov, str(missing_prov))

## Check 6 — Cross-check labels against R (submission_spec.R)

In [ ]:
rscript = shutil.which("Rscript")
if not rscript:
    print("[skip] Rscript not found -- run locally to cross-check labels against R.")
else:
    r_code = "suppressMessages(source('scripts/lib/submission_spec.R')); cat(jsonlite::toJSON(sst$interventions))"
    res = subprocess.run([rscript, "-e", r_code], cwd=ROOT, capture_output=True, text=True)
    assert res.returncode == 0, res.stderr
    r_iv = set(json.loads(res.stdout))
    check("intervention labels == submission_spec.R sst$interventions", set(interventions) == r_iv,
          f"diff={set(interventions) ^ r_iv}")

## Summary

In [ ]:
print("=" * 60)
if failures:
    print(f"BLOCK 2 CONDITIONS: {len(failures)} CHECK(S) FAILED")
    for n, d in failures:
        print("  -", n, "::", d)
else:
    print("BLOCK 2 CONDITIONS: ALL CHECKS PASSED  \u2705")
assert not failures, f"{len(failures)} check(s) failed"